In [ ]:

# Step 64C.5 - Read and validate Silver transactions

df_silver = spark.read.table("LH_Silver.dbo.Silver_SalesTransactions")

print("Total Silver transactions:", df_silver.count())

df_silver.printSchema()

display(df_silver.limit(10))


In [ ]:

from pyspark.sql import functions as F

# Step 64C.6A - Prepare DimProduct

df_dim_product = (
    df_silver
    .select("ProductID")
    .distinct()
    .orderBy("ProductID")
)

print("Distinct products:", df_dim_product.count())

display(df_dim_product.limit(10))


In [ ]:

# Step 64C.6B - Save DimProduct to Gold Lakehouse

(
    df_dim_product.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("LH_Gold.dbo.DimProduct")
)

print("DimProduct successfully saved to LH_Gold.")


In [ ]:

# Step 64C.7A - Prepare DimCustomer

df_dim_customer = (
    df_silver
    .select("CustomerID")
    .distinct()
    .orderBy("CustomerID")
)

print("Distinct customers:", df_dim_customer.count())

print(
    "UNKNOWN customer records:",
    df_dim_customer.filter(F.col("CustomerID") == "UNKNOWN").count()
)

display(df_dim_customer.limit(10))


In [ ]:

# Step 64C.7B - Save DimCustomer to Gold Lakehouse

(
    df_dim_customer.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("LH_Gold.dbo.DimCustomer")
)

print("DimCustomer successfully saved to LH_Gold.")


In [ ]:

# Step 64C.8A - Identify transaction date range

from pyspark.sql import functions as F

df_date_range = df_silver.agg(
    F.min("TransactionDate").alias("MinDate"),
    F.max("TransactionDate").alias("MaxDate"),
    F.countDistinct("TransactionDate").alias("DistinctTransactionDates")
)

display(df_date_range)


In [ ]:

# Step 64C.8B - Generate DimDate DataFrame

from pyspark.sql import functions as F


df_dim_date = (
    spark.sql("""
        SELECT explode(
            sequence(
                to_date('2025-01-01'),
                to_date('2025-12-31'),
                interval 1 day
            )
        ) AS Date
    """)
    .withColumn("DateKey", F.date_format("Date", "yyyyMMdd").cast("int"))
    .withColumn("Year", F.year("Date"))
    .withColumn("Quarter", F.quarter("Date"))
    .withColumn("MonthNumber", F.month("Date"))
    .withColumn("MonthName", F.date_format("Date", "MMMM"))
    .withColumn("DayOfMonth", F.dayofmonth("Date"))
    .withColumn("DayOfWeek", F.dayofweek("Date"))
    .select(
        "DateKey",
        "Date",
        "Year",
        "Quarter",
        "MonthNumber",
        "MonthName",
        "DayOfMonth",
        "DayOfWeek"
    )
    .orderBy("Date")
)

print("Total calendar dates:", df_dim_date.count())

display(df_dim_date.limit(10))


In [ ]:

# Step 64C.8C - Save DimDate to Gold Lakehouse

(
    df_dim_date.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("LH_Gold.dbo.DimDate")
)

print("DimDate successfully saved to LH_Gold.")


In [ ]:

# Step 64C.9A - Prepare FactSales DataFrame

from pyspark.sql import functions as F

# Reload Silver transactions if needed
df_silver = spark.read.table(
    "LH_Silver.dbo.Silver_SalesTransactions"
)

df_fact_sales = (
    df_silver
    .withColumn(
        "DateKey",
        F.date_format("TransactionDate", "yyyyMMdd").cast("int")
    )
    .withColumn(
        "SalesAmount",
        F.round(F.col("Quantity") * F.col("UnitPrice"), 2)
    )
    .select(
        "TransactionID",
        "DateKey",
        "ProductID",
        "CustomerID",
        "Quantity",
        "UnitPrice",
        "SalesAmount"
    )
)

print("Total FactSales rows:", df_fact_sales.count())

df_fact_sales.printSchema()

display(df_fact_sales.limit(10))


In [ ]:

# Step 64C.9B - Validate FactSales

df_dim_date_check = spark.read.table("LH_Gold.dbo.DimDate")
df_dim_product_check = spark.read.table("LH_Gold.dbo.DimProduct")
df_dim_customer_check = spark.read.table("LH_Gold.dbo.DimCustomer")

print("Total FactSales rows:", df_fact_sales.count())

print(
    "Unique transactions:",
    df_fact_sales.select("TransactionID").distinct().count()
)

print(
    "Missing DateKey matches:",
    df_fact_sales.join(
        df_dim_date_check.select("DateKey"),
        "DateKey",
        "left_anti"
    ).count()
)

print(
    "Missing ProductID matches:",
    df_fact_sales.join(
        df_dim_product_check.select("ProductID"),
        "ProductID",
        "left_anti"
    ).count()
)

print(
    "Missing CustomerID matches:",
    df_fact_sales.join(
        df_dim_customer_check.select("CustomerID"),
        "CustomerID",
        "left_anti"
    ).count()
)


In [ ]:

# Step 64C.9C - Save FactSales to Gold Lakehouse

(
    df_fact_sales.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("LH_Gold.dbo.FactSales")
)

print("FactSales successfully saved to LH_Gold.")
